# 13 Reproducible Research — A Minimal Verifiable Workflow

Using the Pine and Cypress Nursing Home Legionella outbreak data to demonstrate a reproducible "from zero to summary" workflow.


In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .


## Importing Packages: Only Import What This Notebook Actually Uses

The first step toward reproducibility is "attaching a manifest" — here we import the three packages the whole notebook needs, all at once: `Path` (handles file paths, cross-platform without surprises), `pandas` (reads data, computes the summary), and `numpy` (Step 4 prints its version number, so the report can be checked against what's installed on your machine).

> **Line-by-line**:
>
> | This line | What it does |
> |---|---|
> | `from pathlib import Path` | Uses object-oriented paths instead of string paths — `Path("data") / "x.csv"` builds the correct slashes on Windows/Mac/Linux alike |
> | `import pandas as pd` | The main workhorse for reading CSVs and computing summary statistics |
> | `import numpy as np` | Step 4 prints its version number and attaches it to the report, so others can check it when reproducing the environment |

> 💡 The import list is itself documentation — the first thing anyone sees when opening the notebook is what's imported, which tells them what this analysis depends on: a second layer of clues beyond `pyproject.toml`.


In [ ]:
# This notebook only needs these three packages, start to finish: Path for paths, pandas to read data, numpy so Step 4 can print its version
from pathlib import Path
import pandas as pd
import numpy as np


## Step 1 — Read the Data + Produce a Summary: The Smallest Unit of Reproducibility

Compress the whole analysis into a single `summary` dict: read from a fixed CSV, and use only **deterministic operations** like `groupby`, `sum`, `mean` (deterministic = the same input always gives the same output, unlike sampling or randomness, which differ every time) to compute a handful of key numbers. This dict is "what someone rerunning this on a different machine should compute, exactly."

> **Line-by-line**:
>
> | This line | What it does |
> |---|---|
> | `path = Path("data/synthetic/legionella_outbreak.csv")` | Pins the input file with a fixed path — the path itself is part of being "reproducible" |
> | `df = pd.read_csv(path)` | Reads the data with zero randomness involved — completely predictable behavior |
> | `df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)` | Derives a column with a fixed rule (anything other than not_ill counts as infected); the rule lives in the code, not in someone's memory of manually tagging cases |
> | `summary = {...}` | Collects every key number into one dict, serving as the "one true answer" for this analysis |

> 💡 **Why emphasize "determinism"**: `groupby(...).ngroups`, `.sum()`, `.mean()` are all pure math operations, with nothing to do with randomness, multi-threaded sort order, timezones, or other "invisible variables" — this is exactly what reproducible research is chasing: eliminating anything that could make the same code and the same data produce different results across two runs.


In [ ]:
# --- Step 1: Read the data and produce a summary ---
path = Path("data/synthetic/legionella_outbreak.csv")
df = pd.read_csv(path)
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)  # Fixed rule: anything other than not_ill counts as infected

summary = {  # This dict is the "one true answer" for this analysis
    "n_residents": len(df),
    "n_zones": df.groupby(["floor", "wing"]).ngroups,
    "n_infected": int(df["infected"].sum()),
    "n_deaths": int((df["outcome"] == "dead").sum()),
    "attack_rate": f"{df['infected'].mean():.1%}",
    "cfr": f"{(df['outcome'] == 'dead').sum() / df['infected'].sum():.1%}",
}

print("=== Outbreak Summary ===")
for k, v in summary.items():
    print(f"  {k}: {v}")

print("\n-> This dict is our minimal verifiable result")
print("-> Anyone on any machine should get the exact same numbers")


## Step 2 — Reproducibility Checklist: Confirm the "Foundation" Is There First

Before checking the data logic, confirm that a handful of environment files exist — `uv.lock` pins package versions, the data file must be present, `pyproject.toml` defines the project structure, and the `tests/` directory can run tests. This step isn't analyzing the data — it's analyzing "your analysis environment."

> **Line-by-line**:
>
> | This line | What it does |
> |---|---|
> | `checks = {...}` | Stores each check as "description: boolean" in a dict, so they can be printed one by one |
> | `_P("uv.lock").exists()` | Confirms the lock file exists — without it, package versions could vary from person to person |
> | `all_pass = all(checks.values())` | If even one item is False, the whole thing doesn't count as "reproducible" |

> ⚠️ This checklist only checks "does the file exist," not "is the version correct" — a truly rigorous CI (see `.github/workflows/ci.yml`) also runs `uv sync` to actually install the versions pinned in `uv.lock`, then runs the tests.


In [ ]:
# --- Step 2: Reproducibility checklist ---
from pathlib import Path as _P  # Avoid shadowing the Path imported above

checks = {  # Only checks "does the file exist," not "is the version correct"
    "uv.lock exists": _P("uv.lock").exists(),
    "data file exists": _P("data/synthetic/legionella_outbreak.csv").exists(),
    "pyproject.toml exists": _P("pyproject.toml").exists(),
    "tests/ directory exists": _P("tests").is_dir(),
}

print("=== Reproducibility Checklist ===")
for item, ok in checks.items():
    status = "✓" if ok else "✗"
    print(f"  [{status}] {item}")

all_pass = all(checks.values())
print(f"\n-> {'All checks passed! The environment is reproducible' if all_pass else 'Some checks failed and need fixing'}")


## Step 3 — Write the Summary to a File: Leave Evidence Behind for "The Numbers This Run Produced"

A result that's only printed to the screen has to be rerun from scratch the next time you want to compare it. Save `summary` as both CSV (easy to open in Excel/pandas) and JSON (preserves the original types, so an integer doesn't turn into a string) — from then on, diffing these files tells you whether the result has drifted.

> **Line-by-line**:
>
> | This line | What it does |
> |---|---|
> | `output_path.mkdir(parents=True, exist_ok=True)` | Ensures the output folder exists; `exist_ok=True` means rerunning doesn't raise an error |
> | `summary_df.to_csv(..., index=False)` | Saves as CSV, easy to read with a spreadsheet or another program |
> | `json.dump(summary, f, ensure_ascii=False, indent=2)` | Saves as JSON, where types (int/str) don't all collapse into strings the way CSV does |

> 🧭 CSV is easy to read but "blurs types" (19 might read back as the string "19"); JSON preserves types precisely but needs extra code to parse — saving both covers "for humans to read" and "for programs to read."


In [ ]:
# --- Step 3: Write the summary to CSV ---
import json

# Method 1: save as CSV
summary_df = pd.DataFrame([summary])
output_path = Path("data/processed")
output_path.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(output_path / "summary.csv", index=False)

# Method 2: save as JSON (preserves types)
with open(output_path / "summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("=== Output Files ===")
print(f"  CSV: {output_path / 'summary.csv'}")
print(f"  JSON: {output_path / 'summary.json'}")
print("\n-> Next time you verify, compare these files to see if the results match")


## Step 4 — Record the Environment Version: Write Down "Which Machine Ran This Analysis"

The same code isn't guaranteed to produce identical results on different versions of Python / pandas (package behavior occasionally changes across versions). Print the version numbers and attach them to the report — when someone else fails to reproduce your results, comparing this table is the first thing to check.

> **Line-by-line**:
>
> | This line | What it does |
> |---|---|
> | `sys.version.split()[0]` | Pulls out a clean Python version number (strips build info and other noise) |
> | `platform.platform()` | The operating system and architecture, e.g. Linux/Windows, x86_64/arm64 |
> | `pd.__version__` / `np.__version__` | The key package versions, which should match what's pinned in `uv.lock` |

> 💡 Having `uv.lock` alone isn't the full picture — `uv.lock` guarantees "reinstalling installs the same versions," while what's printed here is "which version was actually installed when this ran this time." In principle the two should match; if they don't (say, a manual `pip install` overrode something), this table is what catches it.


In [ ]:
# --- Step 4: Record version information ---
import sys
import platform

env_info = {  # Compare against the versions pinned in uv.lock, confirming the "actual runtime environment" matches the "locked versions"
    "python_version": sys.version.split()[0],
    "platform": platform.platform(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
}

print("=== Environment Version Info ===")
for k, v in env_info.items():
    print(f"  {k}: {v}")

print("\n-> Attach the version info to your report so others can reproduce your environment")
print("-> uv.lock can automatically pin every package version for you")


## Step 5 — Random Seed = Determinism: The Easiest Reproducibility Trap to Overlook

The first four steps used no randomness at all, since reading data and computing means are both deterministic operations. But the moment "randomness" shows up in an analysis — like `train_test_split` when training a model in Ch10, `torch.manual_seed` in Ch11, or any `np.random` sampling — failing to fix the seed means every run is a different experiment. Here, a minimal example proves it: **the same seed → the same random numbers; no seed set → different every time**.

> **Line-by-line**:
>
> | This line | What it does |
> |---|---|
> | `rng = np.random.default_rng(seed)` | Creates an independent random number generator; the same `seed` always produces the same sequence of random numbers |
> | `rng.integers(0, 100, 5)` | Draws 5 integers between 0-99, simulating any "random sampling" action |
> | Calling `sample(42)` twice | Runs with the same seed twice, to verify whether the output comes out identical |
> | `sample()` (no seed given) | Lets the random number generator seed itself from system entropy, so every run differs |

> 🎲 **This is exactly why Ch10's `train_test_split(..., random_state=42)` and Ch11's `torch.manual_seed(42)` both fix the seed by hand** — without a fixed seed, the model's train/test split and weight initialization would differ every run, and the same code would produce a different accuracy on two runs, making it look like the code is broken when really someone just forgot to fix the randomness.


In [ ]:
# --- Step 5: Random seed = determinism ---
def sample(seed=None):
    rng = np.random.default_rng(seed)
    return rng.integers(0, 100, 5)


print("seed=42, run 1:", sample(42))
print("seed=42, run 2:", sample(42), "-> exactly the same (reproducible)")
print("no seed set   :", sample(), "-> different every time (not reproducible)")


## Step 6 — Data Column Contract (Schema Contract): Catch It Before Downstream Analysis Breaks

Being reproducible isn't just "it reruns this time" — it also has to guarantee "it'll still be the same data structure the next time it runs." If the health department's system renames a column, `clinical_severity` picks up a new category, or `age` picks up negative values, every downstream analysis will quietly compute the wrong thing without ever throwing an error. Here, the raw data is read again, and `assert` statements spell out explicitly "what I'm assuming about this data" — required columns must exist, category values must fall within a known range, numeric values must be sane — and if any one of them fails, execution stops immediately instead of letting bad data quietly flow into Step 1's summary.

> **Line-by-line**:
>
> | This line | What it does |
> |---|---|
> | `REQUIRED_COLUMNS = {...}` | Lists the required columns this analysis depends on, as the first clause of the contract |
> | `missing_cols = REQUIRED_COLUMNS - set(raw.columns)` | Uses a set difference to find missing columns, listing all of them at once instead of checking one by one |
> | `assert not missing_cols, f"..."` | Stops immediately if columns are missing, with an error message that tells you exactly which ones |
> | `set(raw["clinical_severity"].dropna().unique()) - VALID_SEVERITY` | Checks whether a category value has "escaped" the known range with a new, unrecognized category |
> | `raw["age"].between(0, 120).all()` | Checks whether a numeric column has any value outside a reasonable range (e.g., negative, or an extra zero typed in) |

> ⚠️ This kind of check is called a **schema contract** (data column contract) or data validation in the data science world — production projects often automate it with packages like `pandera` or `great_expectations` and wire it into the pipeline. Here, plain `assert` statements demonstrate the core idea, and the point is "assume the data might break, and write assertions to confirm it hasn't," rather than waiting until the analysis results look weird before going back to hunt for the data problem.


In [ ]:
# --- Step 6: Data column contract (schema contract) ---
raw = pd.read_csv(path)  # Read the raw data again, independent of any transformations done in earlier steps

REQUIRED_COLUMNS = {
    "case_id", "age", "sex", "floor", "wing", "room",
    "clinical_severity", "outcome",
    "symptom_onset_date", "hospitalized", "lab_confirmed",
}
VALID_SEVERITY = {"not_ill", "asymptomatic", "mild", "moderate", "severe"}
VALID_OUTCOME = {"survived", "dead"}

missing_cols = REQUIRED_COLUMNS - set(raw.columns)
assert not missing_cols, f"Missing required columns: {missing_cols}"

unexpected_severity = set(raw["clinical_severity"].dropna().unique()) - VALID_SEVERITY
assert not unexpected_severity, f"clinical_severity has unexpected categories: {unexpected_severity}"

unexpected_outcome = set(raw["outcome"].dropna().unique()) - VALID_OUTCOME
assert not unexpected_outcome, f"outcome has unexpected categories: {unexpected_outcome}"

assert pd.api.types.is_numeric_dtype(raw["age"]), "age column should be numeric dtype"
assert raw["age"].between(0, 120).all(), "age has unreasonable values (outside 0-120)"

print("✅ schema OK - columns, dtypes, and value ranges all as expected")
print(f"   Columns: {len(raw.columns)}, Rows: {len(raw)}")


## Summary

| Step | Skill Learned |
|------|------------|
| Read + summarize | Produce a minimal verifiable result with a dict |
| Checklist | Confirm all environment files are present |
| Save output | Preserve results as CSV / JSON for comparison |
| Version info | Record Python / package versions |
| Random seed | Use a fixed seed so even random processes are reproducible |
| Column contract | Use assert to catch data structure changes early |

**The three pillars of reproducibility**:
1. **Data**: a fixed input file (`legionella_outbreak.csv`) + a schema contract as a gatekeeper
2. **Code**: version control (git commit)
3. **Environment**: pinned packages (`uv.lock`) + a fixed random seed

In the next chapter (Ch14), we'll integrate all these skills into one complete real-world case study.
